In [ ]:
from pathlib import Path
import pandas as pd
from collections import Counter
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from sentinel_flood_mapper.data import remap_labels
from notebooks_utils import check_image_nodata


Paths

In [ ]:
data_root = Path("../data/raw")

Check directory structure

In [ ]:
for item in sorted(data_root.rglob("*")):
    # Print directories and files up to 3 levels deep
    depth = len(item.relative_to(data_root).parts)
    if depth <= 3:
        indent = "   " * (depth - 1)
        if item.is_dir():
            print(f"{indent}[{item.name}/]")
        else:
            print(f"{indent}{item.name}")

In [ ]:
sturm_dirs = {
    "S1 images": Path(data_root / "STURM/Sentinel1/S1"),
    "S1 floodmaps": Path(data_root / "STURM/Sentinel1/Floodmaps"),
    "S2 images": Path(data_root / "STURM/Sentinel2/S2"),
    "S2 floodmaps": Path(data_root / "STURM/Sentinel2/Floodmaps")
}

for name, path in sturm_dirs.items():
    files = list(path.glob("*"))
    extensions = set(f.suffix for f in files if f.is_file())
    print(f"{name}: {len(files)} files | extensions: {extensions}")

Check Sen1Floods11 splits CSVs

In [ ]:
splits_dir = Path(data_root / "Sen1Floods11/splits")

for csv_file in sorted(splits_dir.glob("*.csv")):
    df = pd.read_csv(csv_file)
    print(f"---{csv_file.name}---")
    print(f"Shape: {df.shape}")
    print(df.head(3))
    print()

In [ ]:
s1hand = Path(data_root / "Sen1Floods11/S1Hand")
files = list(s1hand.glob("*.tif"))
print(f"S1Hand files: {len(files)}")

Inspect STURM metadata CSV

In [ ]:
meta = pd.read_csv(data_root / "STURM/Sentinel1_metadata.csv")
print(f"Shape: {meta.shape}")
print(f"\nColumns: {list(meta.columns)}")
print(f"\nFirst 3 rows: ")
print(meta.head(3))
print(f"\nUnique events: {meta['event_id'].nunique() if 'event_id' in meta.columns else 'column name unknown'}")

In [ ]:
# ems_code is the flood event, aoi_code is a sub-area within it
print(f"Unique ems_code (flood events): {meta['ems_code'].nunique()}")
print(f"Unique ems_code + aoi_code combinations: {meta.groupby(['ems_code', 'aoi_code']).ngroups}")
print(f"\nChips per ems_code:")
print(meta.groupby('ems_code')['tile_id'].count().sort_values(ascending=False))
print(f"\nCountries represented:")
print(meta['country'].value_counts())

Inspect Sen1Floods11 country split

In [ ]:
label_dir = Path(data_root / "Sen1Floods11/LabelHand")
files = list(label_dir.glob("*.tif"))

# extract event name from filename e.g. Bolivia_103757_LabelHand.tif -> Bolivia
events = [f.name.split("_")[0] for f in files]
counts = Counter(events)

print(f"Total hand-labelled chips: {len(files)}")
print(f"\nChips per event:")
for event, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {event}: {count}")

In [ ]:
# check Pakistan and Spain events in STURM
overlap_countries = ['Pakistan', 'Spain']
overlap = meta[meta['country'].isin(overlap_countries)][['ems_code', 'country', 'floodmap_date']].drop_duplicates()
print(overlap.to_string())

In [ ]:
# the metadata geojson contains event dates for sen1floods11
meta_path = Path(data_root / "Sen1Floods11")

# check if metadata geojson exists
geojson_files = list(meta_path.rglob("*.geojson"))
json_files = list(meta_path.rglob("*.json"))
print("geojson files:", geojson_files)
print("json files:", json_files)

Summary of data so far...

In [ ]:
# Sen1Floods11 event chip counts
label_dir = Path(data_root / "Sen1Floods11/LabelHand")
sen1_counts = Counter(f.name.split("_")[0] for f in label_dir.glob("*.tif"))

# STURM event chip counts
meta = pd.read_csv(data_root / "STURM/Sentinel1_metadata.csv")
sturm_counts = meta.groupby('ems_code')['tile_id'].count()

print("=== Sen1Floods11 hand-labelled chips per event ===")
for event, count in sorted(sen1_counts.items(), key=lambda x: -x[1]):
    print(f"  {event}: {count}")

print(f"\n  TOTAL: {sum(sen1_counts.values())}")

print("\n=== STURM chips per event ===")
print(sturm_counts.sort_values(ascending=False).to_string())
print(f"\n  TOTAL: {sturm_counts.sum()}")

print(f"\n=== Combined total ===")
print(f"  {sum(sen1_counts.values()) + sturm_counts.sum()} chips across {len(sen1_counts) + sturm_counts.nunique()} events")

print("\n=== Potential overlap to keep in same split ===")
print("  Spain (Sen1Floods11) <-> EMSR397 (STURM) — September/October 2019, same event")
print("  Pakistan (Sen1Floods11) <-> EMSR629 (STURM) — different years, no overlap")

In [ ]:
# build a unified event table
meta = pd.read_csv(data_root / "STURM/Sentinel1_metadata.csv")

rows = []

# Sen1Floods11 events
label_dir = Path(data_root / "Sen1Floods11/LabelHand")
sen1_counts = Counter(f.name.split("_")[0] for f in label_dir.glob("*.tif"))
for event, count in sen1_counts.items():
    rows.append({
        'dataset': 'Sen1Floods11',
        'event_id': event,
        'country': event,  # event name is country in sen1floods11
        'chip_count': count,
        'note': 'possible overlap with EMSR397' if event == 'Spain' else ''
    })

# STURM events
sturm_summary = meta.groupby('ems_code').agg(
    chip_count=('tile_id', 'count'),
    country=('country', 'first'),
    event_type=('event_type', 'first'),
    date=('floodmap_date', 'first')
).reset_index().rename(columns={'ems_code': 'event_id'})
sturm_summary['dataset'] = 'STURM'
sturm_summary['note'] = sturm_summary['event_id'].map(
    lambda x: 'possible overlap with Sen1Floods11 Spain' if x == 'EMSR397' else ''
)

combined = pd.concat([
    pd.DataFrame(rows),
    sturm_summary[['dataset', 'event_id', 'country', 'chip_count', 'note']]
], ignore_index=True).sort_values('chip_count', ascending=False)

print(combined.to_string(index=False))
print(f"\nTotal chips: {combined['chip_count'].sum()}")

Check proposed data split to find a balance of events for train/val/test

In [ ]:
# check running totals for each proposed split
proposed = {
    'train': [
        'EMSR429','EMSR342','EMSR352','EMSR424','EMSR517','EMSR261',
        'EMSR265','EMSR275','EMSR471','EMSR438','EMSR479','EMSR502',
        'EMSR419','EMSR292','EMSR501','EMSR332','EMSR324','EMSR421',
        'EMSR550','EMSR444','EMSR554','EMSR293','EMSR460','EMSR468',
        'EMSR283','EMSR498','EMSR268','EMSR520','EMSR518','EMSR411',
        'EMSR496','EMSR333','EMSR414','EMSR260',
        'USA','India','Paraguay','Ghana','Mekong','Pakistan','Nigeria'
    ],
    'val': [
        'EMSR629','EMSR470','EMSR567','EMSR637','EMSR407','EMSR358',
        'EMSR507','EMSR467','EMSR551',
        'Sri-Lanka','Somalia'
    ],
    'test': [
        'EMSR279','EMSR416','EMSR397',
        'Spain','Bolivia'
    ]
}

# chip counts lookup
chip_counts = dict(zip(combined['event_id'], combined['chip_count']))

for split, events in proposed.items():
    total = sum(chip_counts[e] for e in events)
    pct = total / 22048 * 100
    print(f"{split}: {total} chips ({pct:.1f}%)  —  {len(events)} events")

In [ ]:
proposed = {
    'train': [
        'EMSR429','EMSR342','EMSR352','EMSR424','EMSR517','EMSR261',
        'EMSR265','EMSR275','EMSR471','EMSR438','EMSR479','EMSR502',
        'EMSR419','EMSR292','EMSR501','EMSR332','EMSR324','EMSR421',
        'EMSR550','EMSR444','EMSR554','EMSR293','EMSR460','EMSR468',
        'EMSR283','EMSR498','EMSR268','EMSR520','EMSR518','EMSR411',
        'EMSR496','EMSR333','EMSR414','EMSR260',
        'USA','India','Paraguay','Ghana','Mekong','Pakistan','Nigeria'
    ],
    'val': [
        'EMSR567','EMSR407','EMSR358',
        'EMSR507','EMSR467','EMSR551',
        'Sri-Lanka','Somalia'
    ],
    'test': [
        'EMSR629','EMSR470','EMSR637',
        'EMSR279','EMSR416','EMSR397',
        'Spain','Bolivia'
    ]
}

# chip counts lookup
chip_counts = dict(zip(combined['event_id'], combined['chip_count']))

for split, events in proposed.items():
    total = sum(chip_counts[e] for e in events)
    pct = total / 22048 * 100
    continent_check = [f"{e}({chip_counts[e]})" for e in events]
    print(f"{split}: {total} chips ({pct:.1f}%)  —  {len(events)} events")
    print(f"  {', '.join(continent_check)}")
    print()

In [ ]:
proposed = {
    'train': [
        'EMSR429','EMSR342','EMSR352','EMSR424','EMSR517','EMSR261',
        'EMSR265','EMSR275','EMSR471','EMSR438','EMSR479','EMSR502',
        'EMSR419','EMSR292','EMSR501','EMSR332','EMSR324','EMSR421',
        'EMSR550','EMSR444','EMSR554','EMSR293','EMSR460','EMSR468',
        'EMSR283','EMSR498','EMSR268','EMSR520','EMSR518','EMSR411',
        'EMSR496','EMSR333','EMSR414','EMSR260',
        'USA','India','Paraguay','Ghana','Mekong','Pakistan','Nigeria'
    ],
    'val': [
        'EMSR470','EMSR567','EMSR407','EMSR358',
        'EMSR507','EMSR467','EMSR551',
        'Sri-Lanka','Somalia'
    ],
    'test': [
        'EMSR629','EMSR637',
        'EMSR279','EMSR416','EMSR397',
        'Spain','Bolivia'
    ]
}

chip_counts = dict(zip(combined['event_id'], combined['chip_count']))

for split, events in proposed.items():
    total = sum(chip_counts[e] for e in events)
    pct = total / 22048 * 100
    continent_check = [f"{e}({chip_counts[e]})" for e in events]
    print(f"{split}: {total} chips ({pct:.1f}%)  —  {len(events)} events")
    print(f"  {', '.join(continent_check)}")
    print()

Load and inspect a sample chip from each dataset

In [ ]:
# Sen1Floods11 sample chip
s1_path = Path(data_root / "Sen1Floods11/S1Hand/Ghana_103272_S1Hand.tif")
label_path = Path(data_root / "Sen1Floods11/LabelHand/Ghana_103272_LabelHand.tif")

with rasterio.open(s1_path) as src:
    s1_image = src.read()
    s1_meta = src.meta
    s1_nodata = src.nodata

print("--- Sen1Floods11 S1Hand ---")
print(f" Shape (bands, h, w): {s1_image.shape}")
print(f"Dtype: {s1_image.dtype}")
print(f"Nodata value: {s1_nodata}")
print(f"Min / Max: {s1_image.min():.4f} / {s1_image.max():.4f}")
print(f"CRS: {s1_meta['crs']}")

with rasterio.open(label_path) as src:
    label_image = src.read(1)
    label_nodata = src.nodata

print("\n--- Sen1Floods11 LabelHand ---")
print(f" Shape (h, w): {label_image.shape}")
print(f"Dtype: {label_image.dtype}")
print(f"Unique values: {np.unique(label_image)}")
print(f"Nodata value: {label_nodata}")


In [ ]:
# STURM sample chip
sturm_s1_dir = Path(data_root / "STURM/Sentinel1/S1")
sturm_label_dir = Path(data_root / "STURM/Sentinel1/Floodmaps")

sturm_s1_file = sorted(sturm_s1_dir.glob("*.tif"))[0]
sturm_label_file = sturm_label_dir / sturm_s1_file.name

with rasterio.open(sturm_s1_file) as src:
    sturm_image = src.read()
    sturm_meta = src.meta
    sturm_nodata = src.nodata

print(f"\n--- STURM S1 ({sturm_s1_file.name}) ---)")
print(f" Shape (bands, h, w): {sturm_image.shape}")
print(f"Dtype: {sturm_image.dtype}")
print(f"Nodata value: {sturm_nodata}")
print(f"Min / Max: {sturm_image.min():.4f} / {sturm_image.max():.4f}")
print(f"CRS: {sturm_meta['crs']}")

with rasterio.open(sturm_label_file) as src:
    sturm_label = src.read(1)
    sturm_label_nodata = src.nodata

print("\n --- STURM Floodmap ---")
print(f"Shape (h, w): {sturm_label.shape}")
print(f"Dtype: {sturm_label.dtype}")
print(f"Uniq values: {np.unique(sturm_label)}")


Check STURM labels against metadata

In [ ]:
meta = pd.read_csv(data_root / "STURM/Sentinel1_metadata.csv")
print(meta.columns.tolist())
print(meta.head(3))

In [ ]:
# Check if there is any documentation of label classes in floodmap files themselves
sturm_lbl_file = Path(data_root / "STURM/Sentinel1/Floodmaps/EMSR260_02VIADANA_2_1_2_2.tif")
with rasterio.open(sturm_lbl_file) as src:
    print(f"Tags: {src.tags()}")
    print(f"Band descriptions: {src.descriptions}")
    print(f"Profile: {src.profile}")

In [ ]:
floodmap_dir = Path(data_root / "STURM/Sentinel1/Floodmaps")
all_values = set()

files = list(floodmap_dir.glob("*.tif"))
print(f"Total floodmap files found: {len(files)}")

for f in files[:200]:
    with rasterio.open(f) as src:
        data = src.read(1)
        all_values.update(np.unique(data).tolist())

print(f"All unique label values across 200 STURM floodmaps: {sorted(all_values)}")

In [ ]:
floodmap_dir = Path(data_root / "STURM/Sentinel1/Floodmaps")
files = list(floodmap_dir.glob("*.tif"))

value_counts = Counter()

for f in files[:200]:
    with rasterio.open(f) as src:
        data = src.read(1)
        for val, count in zip(*np.unique(data, return_counts=True)):
            value_counts[int(val)] += int(count)

total = sum(value_counts.values())
print("Value  |  Count      |  % of pixels")
print("-" * 40)
for val in sorted(value_counts.keys()):
    count = value_counts[val]
    print(f"  {val:3d}  |  {count:10d}  |  {count/total*100:.2f}%")

In [ ]:
floodmap_dir = Path(data_root / "STURM/Sentinel1/Floodmaps")
files = list(floodmap_dir.glob("*.tif"))

value_counts = Counter()

for f in files:
    with rasterio.open(f) as src:
        data = src.read(1)
        for val, count in zip(*np.unique(data, return_counts=True)):
            value_counts[int(val)] += int(count)

total = sum(value_counts.values())
print("Value  |  Count      |  % of pixels")
print("-" * 40)
for val in sorted(value_counts.keys()):
    count = value_counts[val]
    print(f"  {val:3d}  |  {count:10d}  |  {count/total*100:.4f}%")

Visualise sample chip from each dataset with its label mask

In [ ]:
# Paths
s1_path  = Path(data_root / "Sen1Floods11/S1Hand/Ghana_103272_S1Hand.tif")
lbl_path = Path(data_root / "Sen1Floods11/LabelHand/Ghana_103272_LabelHand.tif")

sturm_s1_dir  = Path(data_root / "STURM/Sentinel1/S1")
sturm_lbl_dir = Path(data_root / "STURM/Sentinel1/Floodmaps")
sturm_s1_file  = sorted(sturm_s1_dir.glob("*.tif"))[0]
sturm_lbl_file = sturm_lbl_dir / sturm_s1_file.name

In [ ]:
# Load
with rasterio.open(s1_path) as src:
    s1 = src.read()        # shape (2, 512, 512)

with rasterio.open(lbl_path) as src:
    lbl = src.read(1)      # shape (512, 512)

with rasterio.open(sturm_s1_file) as src:
    sturm_s1 = src.read()  # shape (2, 128, 128)

with rasterio.open(sturm_lbl_file) as src:
    sturm_lbl = src.read(1)

In [ ]:
# Label remapping
sen1_map  = {-1: 255, 0: 0, 1: 1}
sturm_map = {0: 0, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 99: 255}

lbl_remapped       = remap_labels(lbl, sen1_map)
sturm_lbl_remapped = remap_labels(sturm_lbl, sturm_map)

In [ ]:
# Visualisation
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Sen1Floods11 row
axes[0, 0].imshow(s1[0], cmap="gray")
axes[0, 0].set_title("Sen1Floods11 — VV band")

axes[0, 1].imshow(s1[1], cmap="gray")
axes[0, 1].set_title("Sen1Floods11 — VH band")

masked_lbl = np.ma.masked_where(lbl_remapped == 255, lbl_remapped)
axes[0, 2].imshow(s1[0], cmap="gray")
axes[0, 2].imshow(masked_lbl, cmap="Blues", alpha=0.5, vmin=0, vmax=1)
axes[0, 2].set_title("Sen1Floods11 — VV + flood mask")

# STURM row
axes[1, 0].imshow(sturm_s1[0], cmap="gray")
axes[1, 0].set_title("STURM — VV band")

axes[1, 1].imshow(sturm_s1[1], cmap="gray")
axes[1, 1].set_title("STURM — VH band")

masked_sturm = np.ma.masked_where(sturm_lbl_remapped == 255, sturm_lbl_remapped)
axes[1, 2].imshow(sturm_s1[0], cmap="gray")
axes[1, 2].imshow(masked_sturm, cmap="Blues", alpha=0.5, vmin=0, vmax=1)
axes[1, 2].set_title("STURM — VV + flood mask")

for ax in axes.flat:
    ax.axis("off")

plt.suptitle("Dataset sample comparison", fontsize=14)
plt.tight_layout()
plt.show()

Check label value distribution and class balance across both datasets

In [ ]:
sen1_map  = {-1: 255, 0: 0, 1: 1}
sturm_map = {0: 0, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 99: 255}

In [ ]:
# Sen1Floods11
lbl_dir = Path(data_root / "Sen1Floods11/LabelHand")
sen1_counts = Counter()

for f in sorted(lbl_dir.glob("*.tif")):
    with rasterio.open(f) as src:
        arr = remap_labels(src.read(1), sen1_map)
        for val, count in zip(*np.unique(arr, return_counts=True)):
            sen1_counts[int(val)] += int(count)

sen1_total = sum(sen1_counts.values())
print("--- Sen1Floods11 LabelHand (remapped) ---")
print(f"  Total pixels: {sen1_total:,}")
for val, label in [(0, 'non-water'), (1, 'water'), (255, 'ignore/nodata')]:
    count = sen1_counts[val]
    print(f"  {val:3d} ({label:15s}): {count:12,}  |  {count/sen1_total*100:.2f}%")

In [ ]:
# Sturm
sturm_lbl_dir = Path(data_root / "STURM/Sentinel1/Floodmaps")
sturm_counts = Counter()

for f in sorted(sturm_lbl_dir.glob("*.tif")):
    with rasterio.open(f) as src:
        arr = remap_labels(src.read(1), sturm_map)
        for val, count in zip(*np.unique(arr, return_counts=True)):
            sturm_counts[int(val)] += int(count)

sturm_total = sum(sturm_counts.values())
print("\n--- STURM Floodmaps (remapped) ---")
print(f"  Total pixels: {sturm_total:,}")
for val, label in [(0, 'non-water'), (1, 'water'), (255, 'ignore/nodata')]:
    count = sturm_counts[val]
    print(f"  {val:3d} ({label:15s}): {count:12,}  |  {count/sturm_total*100:.2f}%")

In [ ]:
# Combined
combined_total = sen1_total + sturm_total
combined_water = sen1_counts[1] + sturm_counts[1]
combined_nonwater = sen1_counts[0] + sturm_counts[0]
combined_ignore = sen1_counts[255] + sturm_counts[255]

print("\n--- Combined (after remapping) ---")
print(f"  Total pixels:  {combined_total:,}")
print(f"  non-water:     {combined_nonwater:,}  |  {combined_nonwater/combined_total*100:.2f}%")
print(f"  water:         {combined_water:,}  |  {combined_water/combined_total*100:.2f}%")
print(f"  ignore/nodata: {combined_ignore:,}  |  {combined_ignore/combined_total*100:.2f}%")
print(f"\n  Water:non-water ratio: 1:{combined_nonwater/combined_water:.1f}")

 Check for nodata and nan values in SAR images

In [ ]:
print("--- Sen1Floods11 S1Hand ---")
check_image_nodata(data_root / "Sen1Floods11/S1Hand", "*.tif", n_sample=50)

print("\n--- STURM Sentinel1 S1 ---")
check_image_nodata(data_root / "STURM/Sentinel1/S1", "*.tif", n_sample=50)

In [ ]:
sturm_s1_dir = Path(data_root / "STURM/Sentinel1/S1")
files = sorted(sturm_s1_dir.glob("*.tif"))

mins, maxs, means = [], [], []
for f in files[:200]:
    with rasterio.open(f) as src:
        arr = src.read().astype(np.float32)
    mins.append(float(np.nanmin(arr)))
    maxs.append(float(np.nanmax(arr)))
    means.append(float(np.nanmean(arr)))

print(f"Over 200 files:")
print(f"  Absolute min:  {min(mins):.4f}")
print(f"  Absolute max:  {max(maxs):.4f}")
print(f"  Mean of means: {np.mean(means):.4f}")
print(f"  Min of means:  {min(means):.4f}")
print(f"  Max of means:  {max(means):.4f}")